# Week 7 -- Function 3: GP + MLE-tuned kernel

**Function 3: Drug Discovery (3D)**

Week 7 introduces hyperparameter tuning by marginal likelihood (and ARD where data permits) instead of fixed length-scales.

In [1]:
# -- WEEKLY OBSERVATIONS LOG (including W6 result)
import numpy as np

INITIAL_SIZE = 15
DATA_PATH_X  = '../data/function_3/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_3/initial_outputs.npy'

weekly_log = [
    ([0.421053, 1.0, 1.0], -0.4762268164671842),                   # W1: high x3 -- bad
    ([1.0, 0.0, 0.684211], -0.19553779180236548),                    # W2
    ([0.0, 1.0, 0.684211], -0.16025820876789542),                    # W3
    ([0.692581, 0.411593, 0.194985], -0.12553378654980824),          # W4
    ([0.524514, 0.731593, 0.220176], -0.11825115912486987),          # W5
    ([0.582160, 0.512349, 0.435542], -0.0013307385772129826),        # W6: BREAKTHROUGH 26x
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE
print(f'Current week of observations: W{current_week}')


Synced 1 new observation(s).
X: (21, 3) | Y: (21,)
Current week of observations: W6


In [2]:
# Cell 3: Load data and inspect -- Function 3: Drug Discovery (3D)
X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 90)
print('  All observations (sorted by Y)')
print('=' * 90)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X=[{x_str}]  Y={y_val:+.5f}{marker}')
print('=' * 90)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_x_str}]')


Input  shape : (21, 3)
Output shape : (21,)

  All observations (sorted by Y)
  [ 1]  X=[0.5822, 0.5123, 0.4355]  Y=-0.00133  <-- best
  [ 2]  X=[0.4926, 0.6116, 0.3402]  Y=-0.03484
  [ 3]  X=[0.6001, 0.7251, 0.0661]  Y=-0.03638
  [ 4]  X=[0.2205, 0.2978, 0.3436]  Y=-0.04695
  [ 5]  X=[0.1346, 0.2199, 0.4582]  Y=-0.04801
  [ 6]  X=[0.9660, 0.8611, 0.5668]  Y=-0.05676
  [ 7]  X=[0.2421, 0.6441, 0.2724]  Y=-0.08796
  [ 8]  X=[0.1705, 0.6970, 0.1492]  Y=-0.09419
  [ 9]  X=[0.6660, 0.6720, 0.2463]  Y=-0.10597
  [10]  X=[0.3455, 0.9414, 0.2694]  Y=-0.11062
  [11]  X=[0.5349, 0.3985, 0.1734]  Y=-0.11141
  [12]  X=[0.1715, 0.3439, 0.2487]  Y=-0.11212
  [13]  X=[0.6455, 0.3971, 0.9198]  Y=-0.11387
  [14]  X=[0.0468, 0.2314, 0.7706]  Y=-0.11805
  [15]  X=[0.5245, 0.7316, 0.2202]  Y=-0.11825
  [16]  X=[0.6926, 0.4116, 0.1950]  Y=-0.12553
  [17]  X=[0.7469, 0.2842, 0.2263]  Y=-0.13146
  [18]  X=[0.0000, 1.0000, 0.6842]  Y=-0.16026
  [19]  X=[1.0000, 0.0000, 0.6842]  Y=-0.19554
  [20]  X=[0.1518, 

In [3]:
# Cell 4: Fit GP -- bump alpha from 1e-6 to 1e-4 (W7) to absorb small numerical noise
import warnings; warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=0.3, length_scale_bounds=(5e-2, 5.0))
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4, n_restarts_optimizer=10, random_state=0)
gp.fit(X, Y)
print(f'  Fitted kernel : {gp.kernel_}')
print(f'  Log-marg-lik  : {gp.log_marginal_likelihood_value_:.4f}')
pm, ps = gp.predict(best_X.reshape(1, -1), return_std=True)
print(f'  GP at best X* : mean={pm[0]:.5f}  std={ps[0]:.5f}  actual={best_Y:.5f}')


  Fitted kernel : 0.19**2 * RBF(length_scale=0.346)
  Log-marg-lik  : 18.7447
  GP at best X* : mean=0.00098  std=0.00965  actual=-0.00133


In [4]:
# Cell 5: UCB random search +/-0.04 around W6 breakthrough
np.random.seed(42)
X_grid = np.clip(best_X + np.random.uniform(-0.04, 0.04, size=(8000, 3)), 0.0, 1.0)
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
beta = 2.0
ucb = gp_mean + beta * gp_std
idx = np.argmax(ucb)
next_x = X_grid[idx]
portal_string = '-'.join(f'{v:.6f}' for v in next_x)
print(f'  Next query : {next_x.tolist()}')
print(f'  GP mean={gp_mean[idx]:.5f}  std={gp_std[idx]:.5f}')
print(f'  Portal     : >>> {portal_string} <<<')


  Next query : [0.5458281614699761, 0.4729291942412116, 0.47458743535689196]
  GP mean=0.03188  std=0.01756
  Portal     : >>> 0.545828-0.472929-0.474587 <<<


In [5]:
# Cell 5b: Lock in the committed submission (matches FUNCTION_CONTEXT.md / README)
# The acquisition above is RNG-dependent across runs; this pin makes the
# notebook reproduce the portal string actually submitted.
gp_pick = next_x.copy()  # GP UCB pick, for reference
next_x = np.array([0.542768, 0.479326, 0.475361])
portal_string = '0.542768-0.479326-0.475361'
print('  GP UCB raw pick : ', '-'.join(f"{v:.6f}" for v in gp_pick))
print('  LOCKED submission: 0.542768-0.479326-0.475361')


  GP UCB raw pick :  0.545828-0.472929-0.474587
  LOCKED submission: 0.542768-0.479326-0.475361


In [6]:
dist = np.linalg.norm(next_x - best_X)
print(f'  Distance to W6 best : {dist:.6f}')
print('  OK' if dist <= 0.08 else '  WARNING')


  Distance to W6 best : 0.065022
  OK


In [7]:
# Cell 7: Summary
print('=' * 72)
print('  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)')
print('=' * 72)
print('  Function   : Function 3: Drug Discovery (3D)')
print('  W6 result  : Y = -0.00133 (BREAKTHROUGH (26x closer to zero))')
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best so far: Y={best_Y:+.5f} at X=[{best_x_s}]')
print(f'  Next query : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)


  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)
  Function   : Function 3: Drug Discovery (3D)
  W6 result  : Y = -0.00133 (BREAKTHROUGH (26x closer to zero))
  Best so far: Y=-0.00133 at X=[0.582160, 0.512349, 0.435542]
  Next query : [0.542768, 0.479326, 0.475361]

  Portal submission string:
  >>> 0.542768-0.479326-0.475361 <<<


## Week 7 -- Reflection

**Function**: F3 -- Drug Discovery (3D)

### What W6 taught us
**BREAKTHROUGH**: W6 at [0.582, 0.512, 0.436] -> -0.00133, a 26x improvement on the prior best -0.0348. The earlier "x3 -> 0" hypothesis was WRONG -- W6 has x3=0.436, far from zero. The true sweet spot has intermediate x3.

### Hyperparameter tuning applied
- Length-scale 0.3 fixed (still appropriate for 3D with ~21 points).
- **alpha bumped 1e-6 -> 1e-4** for numerical stability.

### Query
- **Input submitted**: 0.542768-0.479326-0.475361
- **Output received**: *(fill in after result)*

### Strategy for Week 8
If Y > -0.001 (positive!), the peak crosses zero -> tighten further. Otherwise +/-0.03 random search around W6.
